In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import SimpleITK as sitk
import os
import numpy as np
import torchvision.utils as vutils
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import time
import csv

# ---------------------------
# Dataset
# ---------------------------
class Segmentation3DDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = sitk.GetArrayFromImage(sitk.ReadImage(self.image_paths[idx]))
        image = np.expand_dims(image, axis=0)  # [C=1, D, H, W]
        if self.transform:
            image = self.transform(image)
        return torch.from_numpy(image)  # ensure int for CE

# ---------------------------
# Encoder
# ---------------------------
class Encoder3D_GS(nn.Module):
    def __init__(self, input_channels=1, latent_dim=16, n_categories=4):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(input_channels*2, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv3d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv3d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.flatten = nn.Flatten()
        self.latent_dim = latent_dim
        self.n_categories = n_categories
        self.fc = None  # define dynamically in forward

    def forward(self, x, atlas):
        x = torch.cat([x, atlas], dim=1)  # atlas as extra channel
        h = self.conv(x)
        if self.fc is None:
            self.fc = nn.Linear(h.view(h.size(0), -1).size(1), self.latent_dim*self.n_categories).to(h.device)
        h_flat = h.view(h.size(0), -1)
        logits = self.fc(h_flat)
        return logits.view(-1, self.latent_dim, self.n_categories)

# ---------------------------
# Gumbel-Softmax
# ---------------------------
def gumbel_softmax_sample(logits, tau=1.0, hard=False):
    U = torch.rand_like(logits)
    g = -torch.log(-torch.log(U + 1e-10) + 1e-10)
    y = F.softmax((logits + g)/tau, dim=-1)
    if hard:
        y_hard = torch.zeros_like(y)
        y_hard.scatter_(-1, y.argmax(dim=-1, keepdim=True), 1.0)
        y = (y_hard - y).detach() + y
    return y

# ---------------------------
# Decoder
# ---------------------------
class Decoder3D(nn.Module):
    def __init__(self, latent_dim=16, n_categories=4, output_channels=271):
        super().__init__()
        self.latent_dim = latent_dim
        self.n_categories = n_categories
        self.output_channels = output_channels
        self.fc = None  # dynamic
        self.deconv = nn.Sequential(
            nn.ConvTranspose3d(128, 64, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose3d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.Conv3d(32, output_channels, 3, padding=1)
        )

    def forward(self, z, atlas):
        B = z.size(0)
        if self.fc is None:
            # dynamically compute shape
            dummy = torch.zeros(B, self.latent_dim, self.n_categories, device=z.device)
            self.fc = nn.Linear(self.latent_dim*self.n_categories, 128*np.prod(atlas.shape[2:] // 4)).to(z.device)
        h = self.fc(z.view(B, -1))
        D, H, W = atlas.shape[2]//4, atlas.shape[3]//4, atlas.shape[4]//4
        h = h.view(B, 128, D, H, W)
        recon_logits = self.deconv(h)
        return recon_logits

# ---------------------------
# KL Divergence
# ---------------------------
def kl_categorical_uniform(logits, n_categories):
    q_y = F.softmax(logits, dim=-1)
    log_q_y = torch.log(q_y + 1e-10)
    kl = torch.sum(q_y * (log_q_y - torch.log(torch.tensor(1.0/n_categories).to(logits.device))), dim=-1)
    return kl.mean()

# ===== Your consistent color setup =====
base_colors = plt.cm.get_cmap('tab20').colors  # 20 RGBA colors
n_labels = 300
repeated_colors = np.tile(base_colors, (n_labels // 20 + 1, 1))[:n_labels]
cmap = ListedColormap(repeated_colors)
n_labels = 270
norm = BoundaryNorm(np.arange(n_labels + 1), cmap.N)

# Manually set label 0 to white
colors_with_white_bg = cmap.colors
colors_with_white_bg[0] = (1.0, 1.0, 1.0)  # RGB white
cmap = ListedColormap(colors_with_white_bg)

def __write_images(image_outputs, display_image_num, file_name):
    imgs, recons = image_outputs

    # If 3D volumes: [B, D, H, W] → take middle slice
    if imgs.ndim == 4:
        slice_idx = imgs.shape[1] // 2
        imgs = torch.stack([img[slice_idx] for img in imgs[:display_image_num]])
        recons = torch.stack([img[slice_idx] for img in recons[:display_image_num]])
    else:
        imgs = imgs[:display_image_num]
        recons = recons[:display_image_num]

    # Apply discrete colormap and return RGB tensors
    def apply_cmap_rgb(tensor):
        arr = tensor.cpu().numpy().astype(np.int32)
        rgb_list = []
        for img in arr:
            rgb_img = cmap(norm(img))[..., :3]  # drop alpha
            rgb_tensor = torch.from_numpy(rgb_img).permute(2, 0, 1)  # [3, H, W]
            rgb_list.append(rgb_tensor)
        return torch.stack(rgb_list)

    imgs_rgb = apply_cmap_rgb(imgs)
    recons_rgb = apply_cmap_rgb(recons)

    # Create grids
    grid_in = vutils.make_grid(imgs_rgb, nrow=display_image_num, padding=2)
    grid_rec = vutils.make_grid(recons_rgb, nrow=display_image_num, padding=2)

    # Stack vertically
    full_grid = torch.cat([grid_in, grid_rec], dim=1)

    vutils.save_image(full_grid, file_name)

class EarlyStopping:
    def __init__(self, patience=10, verbose=True, save_path="best_model.pth"):
        """
        patience: how many epochs to wait after last improvement
        verbose: print updates
        save_path: where to save the best model
        """
        self.patience = patience
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False
        self.verbose = verbose
        self.save_path = save_path

    def __call__(self, val_loss, model_dict):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            self.save_checkpoint(model_dict)
        else:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, model_dict):
        if self.verbose:
            print(f"Validation loss improved → {self.best_loss:.4f}. Saving model...")
        torch.save(model_dict, self.save_path)



# ---------------------------
# Training Loop
# ---------------------------
def train_gs_vae(train_paths, val_paths, save_folder, batch_size=2, lr=1e-4, num_epochs=1000,
                 latent_dim=16, n_categories=4, n_classes=271, tau_init=1.0, tau_min=0.5,
                 tau_anneal=0.99, save_imgs_freq=5, save_model_freq=5):

    os.makedirs(save_folder + "/test_images", exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = Segmentation3DDataset(train_paths)
    val_dataset = Segmentation3DDataset(val_paths)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

    encoder = Encoder3D_GS(1, latent_dim, n_categories).to(device)
    decoder = Decoder3D(latent_dim, n_categories, n_classes).to(device)
    optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()

    early_stopping = EarlyStopping(patience=5, save_path=save_folder+"best_model.pth")
    tau = tau_init

    print("Start training!!")
    for epoch in range(num_epochs):
        start_time = time.time()
        encoder.train(); decoder.train()
        train_loss = 0

        for batch in train_loader:
            batch = batch.to(device)
            atlas_image = sitk.GetArrayFromImage(sitk.ReadImage("../../MDSC689.03-Final-Project/spring term/data/anat_atlases/human_anat_seg_common.nii"))
            atlas_image = np.expand_dims(atlas_image, axis=0)  # [1, D, H, W]
            atlas_tensor = torch.from_numpy(atlas_image).float().to(device)
            atlas_tensor = one_hot_encode(atlas_tensor, n_classes)


            logits = encoder(batch, atlas_tensor)
            z = gumbel_softmax_sample(logits, tau=tau, hard=False)
            recon_logits = decoder(z, atlas_tensor)

            recon_loss = criterion(recon_logits, batch)
            kl_loss = kl_categorical_uniform(logits, n_categories)
            beta = 10
            loss = recon_loss + beta*kl_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Validation
        encoder.eval(); decoder.eval()
        val_loss = 0
        img_to_save, recon_to_save = [], []

        with torch.no_grad():
            for i, val_batch in enumerate(val_loader):
                val_batch = val_batch.to(device)
                atlas_image = sitk.GetArrayFromImage(sitk.ReadImage("../../MDSC689.03-Final-Project/spring term/data/anat_atlases/human_anat_seg_common.nii"))
                atlas_image = np.expand_dims(atlas_image, axis=0)  # [1, D, H, W]
                atlas_tensor = torch.from_numpy(atlas_image).float().to(device)
                atlas_tensor = one_hot_encode(atlas_tensor, n_classes)

                logits = encoder(val_batch, atlas_tensor)
                z = gumbel_softmax_sample(logits, tau=tau, hard=False)
                recon_logits = decoder(z, atlas_tensor)

                recon_loss = criterion(recon_logits, val_batch)
                kl_loss = kl_categorical_uniform(logits, n_categories)
                loss = recon_loss + beta*kl_loss

                val_loss += loss.item()

                # Save images
                if epoch % save_imgs_freq == 0 and i < 16:
                    recon_labels = torch.argmax(recon_logits, dim=1).cpu()
                    recon_to_save.append(recon_labels)
                    img_to_save.append(val_batch.cpu())

        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        elapsed = time.time() - start_time
        print(f"Epoch [{epoch+1}/{num_epochs}], Train: {train_loss:.4f}, Val: {val_loss:.4f}, Time: {elapsed:.2f}s, Tau: {tau:.3f}")

        # Anneal tau
        tau = max(tau_min, tau*tau_anneal)

        # Save CSV log
        with open(os.path.join(save_folder, "loss_log.csv"), "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([epoch, train_loss, val_loss])

        # Save images
        if epoch % save_imgs_freq == 0:
            img_stack = torch.stack(img_to_save).squeeze()
            recon_stack = torch.stack(recon_to_save)
            __write_images([img_stack, recon_stack], display_image_num=16,
                           file_name=save_folder+f"/test_images/recons_epoch_{epoch}.png")

        # Save checkpoint & early stopping
        if epoch % save_model_freq == 0:
            checkpoint = {
                "epoch": epoch,
                "encoder_state_dict": encoder.state_dict(),
                "decoder_state_dict": decoder.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_loss": train_loss,
                "val_loss": val_loss
            }
            early_stopping(val_loss, checkpoint)
            if early_stopping.early_stop:
                print("Early stopping triggered. Training stopped.")
                break

def one_hot_encode(labels, n_classes):
    """
    labels: [D,H,W] or [B,D,H,W], integer label values
    returns: [B, n_classes, D, H, W]
    """
    if labels.ndim == 3:
        labels = labels.unsqueeze(0)  # add batch dim
    B, D, H, W = labels.shape
    one_hot = F.one_hot(labels.long(), num_classes=n_classes)  # [B,D,H,W,n_classes]
    one_hot = one_hot.permute(0, 4, 1, 2, 3).float()            # [B,n_classes,D,H,W]
    return one_hot


# Dataset paths
train_folder = "./datasets/3d_anat_align/human_train/"
train_paths = [os.path.join(train_folder, f) for f in os.listdir(train_folder)]
val_folder = "./datasets/3d_anat_align/human_test/"
val_paths = [os.path.join(val_folder, f) for f in os.listdir(val_folder)]
val_paths.sort()

# Save folder
save_folder = "./VAE_train/3d_anat/human_train_gs/"
os.makedirs(save_folder, exist_ok=True)

# Call training loop
train_gs_vae(
    train_paths=train_paths,
    val_paths=val_paths,
    save_folder=save_folder,
    batch_size=1,
    lr=1e-4,
    num_epochs=1000,
    latent_dim=16,
    n_categories=4,
    n_classes=271,
    tau_init=1.0,
    tau_min=0.5,
    tau_anneal=0.99,
    save_imgs_freq=5,
    save_model_freq=5
)



/tmp/ipykernel_971879/1396196588.py:113: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  base_colors = plt.cm.get_cmap('tab20').colors  # 20 RGBA colors


Start training!!


OutOfMemoryError: CUDA out of memory. Tried to allocate 8.76 GiB (GPU 0; 23.61 GiB total capacity; 17.56 GiB already allocated; 4.88 GiB free; 17.57 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF